In [11]:
from binance.client import Client
import config
import pandas as pd
import matplotlib.pyplot as plt


client = Client(config.api_key, config.api_secret)

In [12]:
account_info = client.get_account()

In [13]:
btc_price = client.get_symbol_ticker(symbol="BTCUSDT")
print(f"BTC Price: {btc_price['price']}")

BTC Price: 68390.07000000


In [14]:
import requests

url = "https://api.binance.com/api/v3/ticker/24hr"
response = requests.get(url)
tickers = response.json()

# Filter USDT pairs
usdt_pairs = [ticker for ticker in tickers if ticker['symbol'].endswith('USDT')]

# Sort by priceChangePercent
sorted_gainers = sorted(usdt_pairs, key=lambda x: float(x['priceChangePercent']), reverse=True)
sorted_losers = sorted(usdt_pairs, key=lambda x: float(x['priceChangePercent']))

# Top 5 Gainers
print("Top 5 Gainers:")
for ticker in sorted_gainers[:5]:
    print(f"{ticker['symbol']} : {ticker['priceChangePercent']}%")

# Top 5 Losers
print("\nTop 5 Losers:")
for ticker in sorted_losers[:5]:
    print(f"{ticker['symbol']} : {ticker['priceChangePercent']}%")


Top 5 Gainers:
CREAMUSDT : 65.354%
PNTUSDT : 45.228%
FORMUSDT : 30.377%
PHAUSDT : 21.586%
1000CHEEMSUSDT : 19.239%

Top 5 Losers:
BETAUSDT : -64.000%
VIBUSDT : -63.262%
WTCUSDT : -56.540%
CHESSUSDT : -55.183%
ACAUSDT : -51.351%


In [ ]:
df = pd.DataFrame(klines, columns=['Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close time', 'Quote asset volume', 'Number of trades', 'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore'])
df['Timestamp'] = pd.to_datetime(df['Open time'], unit='ms')
df.set_index('Timestamp', inplace=True)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['Close'], label='Close Price')
plt.title(f'{symbol} Price Chart')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

In [ ]:
import asyncio
import matplotlib.pyplot as plt
from datetime import datetime
from binance import AsyncClient, BinanceSocketManager

prices = []
timestamps = []

async def main():
    client = await AsyncClient.create()
    bm = BinanceSocketManager(client)
    ts = bm.trade_socket('BNBBTC')

    async with ts as tscm:
        while True:
            res = await tscm.recv()
            timestamp = res['T']
            formatted_time = datetime.fromtimestamp(timestamp / 1000).strftime('%Y-%m-%d %H:%M:%S')
            price = float(res['p'])

            prices.append(price)
            timestamps.append(formatted_time)

            plt.clf()
            plt.plot(timestamps, prices)
            plt.xlabel('Time')
            plt.ylabel('Price (BTC)')
            plt.title('Live BTC Price Updates')
            plt.xticks(rotation=45)
            plt.pause(0.001)  # Update the plot briefly
            plt.draw()  # Re-draw the plot

if __name__ == "__main__":
    loop = asyncio.get_event_loop()  # Use the existing event loop
    loop.create_task(main())
    plt.ion()
    plt.show()

    try:
        loop.run_until_complete(asyncio.gather(
            main_task,  # Other tasks to run concurrently
            loop.create_task(loop.stop())  # Schedule loop closure
        ))
    except KeyboardInterrupt:
        pass  # Handle Ctrl+C gracefully
    finally:
        loop.close()
